### Generate Dataset

Import Libraries

In [1]:
import pandas as pd
import numpy as np

Set Random Seed (This makes results reproducible)

In [2]:
np.random.seed(42)

Generate Users

In [3]:
n_users = 10000

user_ids = range(1, n_users + 1)

Create Groups

5000 Control

5000 Treatment

In [4]:
variant = ['A'] * 5000 + ['B'] * 5000

Generate Conversions

Control:
8% conversion

In [5]:
control_conversion = np.random.binomial(
    1,
    0.08,
    5000
)

Treatment: 10% conversion

In [6]:
treatment_conversion = np.random.binomial(
    1,
    0.10,
    5000
)

Combine

In [7]:
converted = np.concatenate([
    control_conversion,
    treatment_conversion
])

Create Revenue

Assume:

Purchase Value

₹2000

If not converted:

₹0

In [8]:
revenue = np.where(
    converted == 1,
    2000,
    0
)

Build Dataset

In [9]:
df = pd.DataFrame({
    'user_id': user_ids,
    'variant': variant,
    'converted': converted,
    'revenue': revenue
})

Check

In [10]:
df.head()

,user_id,variant,converted,revenue
0,1,A,0,0
1,2,A,1,2000
2,3,A,0,0
3,4,A,0,0
4,5,A,0,0


Save Dataset

In [11]:
df.to_csv(
    'ab_test_data.csv',
    index=False
)

### Analyze Results

Conversion Rate

In [13]:
summary = df.groupby('variant').agg(
    users=('user_id','count'),
    conversions=('converted','sum'),
    revenue=('revenue','sum')
)

summary['conversion_rate'] = (
    summary['conversions']
    /
    summary['users']
) * 100

summary

,users,conversions,revenue,conversion_rate
variant,,,,
A,5000,393,786000,7.86
B,5000,482,964000,9.64


Calculate Lift

In [14]:
control_rate = summary.loc['A','conversion_rate']

treatment_rate = summary.loc['B','conversion_rate']

lift = (
    treatment_rate
    -
    control_rate
) / control_rate * 100

print(lift)

22.646310432569976


#### Statistical Testing

Import:

In [15]:
from statsmodels.stats.proportion import proportions_ztest

Count Successes

In [16]:
successes = [
    summary.loc['A','conversions'],
    summary.loc['B','conversions']
]

Sample Size

In [17]:
samples = [
    summary.loc['A','users'],
    summary.loc['B','users']
]

Run Test

In [18]:
z_stat, p_value = proportions_ztest(successes, samples)


Results

In [19]:
print(z_stat)
print(p_value)

-3.1497025610511185
0.0016343676471522034


Interpretation

In [20]:
if p_value < 0.05:
    print("Statistically Significant")
else:
    print("Not Significant")

Statistically Significant


#### Business Impact Analysis

Revenue Difference

In [21]:
control_revenue = summary.loc['A','revenue']

treatment_revenue = summary.loc['B','revenue']

incremental_revenue = (
    treatment_revenue
    -
    control_revenue
)

print(incremental_revenue)

178000


Revenue Lift

In [22]:
revenue_lift = (
    incremental_revenue
    /
    control_revenue
)*100

print(revenue_lift)

22.646310432569976
